In [2]:
import torch

In [3]:
tokens = ["Your", "journey", "starts", "with", "one", "step"]
x = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     
   [0.55, 0.87, 0.66], # journey  
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]] # step
)

In [4]:
attentionScore = x @ x.t()
attentionScore

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [5]:
attentionScoreTrilled = torch.tril(attentionScore)
attentionScoreTrilled

tensor([[0.9995, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9544, 1.4950, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9422, 1.4754, 1.4570, 0.0000, 0.0000, 0.0000],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.0000, 0.0000],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.0000],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [6]:
attentionWeight = attentionScoreTrilled/attentionScoreTrilled.sum(dim=1, keepdim=False)
attentionWeight

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9549, 0.6104, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9427, 0.6024, 0.3760, 0.0000, 0.0000, 0.0000],
        [0.4755, 0.3443, 0.2141, 0.1869, 0.0000, 0.0000],
        [0.4578, 0.2886, 0.1846, 0.1315, 0.2300, 0.0000],
        [0.6313, 0.4436, 0.2737, 0.2485, 0.1015, 0.2022]])

In [7]:
context_length = x.shape[0]
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attentionScore.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.9995,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.9544, 1.4950,   -inf,   -inf,   -inf,   -inf],
        [0.9422, 1.4754, 1.4570,   -inf,   -inf,   -inf],
        [0.4753, 0.8434, 0.8296, 0.4937,   -inf,   -inf],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654,   -inf],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [8]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.6)                                   
example = torch.ones(6, 6)                                        
print(dropout(example))

tensor([[2.5000, 2.5000, 0.0000, 2.5000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 2.5000, 0.0000, 2.5000],
        [2.5000, 2.5000, 2.5000, 0.0000, 0.0000, 2.5000],
        [0.0000, 2.5000, 2.5000, 0.0000, 0.0000, 2.5000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 2.5000],
        [0.0000, 2.5000, 0.0000, 2.5000, 2.5000, 0.0000]])


In [9]:
import torch.nn as nn

#The word causal simply means relating to cause and effect. It describes a relationship where one event, action, or variable directly makes another event happen.If event A is causal, it means event A is the reason event B exists or changed.
class CausalAttention(nn.Module):
    def __init__(self, dim_in, dim_out, qkvBias,pValue, context_length):
        super().__init__()
        self.queryWeightFunc = nn.Linear(in_features=dim_in, out_features=dim_out, bias=qkvBias)
        self.keyWeightFunc = nn.Linear(in_features=dim_in, out_features=dim_out, bias=qkvBias)
        self.valueWeightFunc = nn.Linear(in_features=dim_in, out_features=dim_out, bias=qkvBias)
        self.dropOutLayer = nn.Dropout(p=pValue)
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1 )
    
    def forward(self, x):
        query = self.queryWeightFunc(x)
        key = self.keyWeightFunc(x)
        value = self.valueWeightFunc(x)

        attentionScore = query @ key.t()
        maskedAttentionScore = torch.masked_fill(attentionScore, mask=self.mask.bool(), value=-torch.inf)
        maskedAttentionWeight = torch.softmax(maskedAttentionScore/ key.shape[-1]**0.5, dim=1)
        maskedAttentionWeight_dropout =  self.dropOutLayer(maskedAttentionWeight)
        context = maskedAttentionWeight_dropout @ value
        return context

In [ ]:
torch.manual_seed(123)
d_in = x.shape[1]                                            
d_out = 2 
causalAttentionObj = CausalAttention(dim_in=d_in, dim_out=2,qkvBias=False, pValue=0.5, context_length = 6)
causalAttentionObj(x)

tensor([[ 0.0000],
        [ 0.0000],
        [-0.6053],
        [-0.7104],
        [-0.9526],
        [-0.6123]], grad_fn=<MmBackward0>)